In [1]:
from google.colab import drive
#drive.mount('/content/drive')

In [2]:
from pathlib import Path
import pandas as pd


In [3]:
DATA_DIR = Path('/content')

In [4]:
required_files = ['churn_logit1.csv', 'loan_default_logit2.csv']

In [5]:
print("=== 파일 존재 여부 확인 ===")
for f in required_files:
    file_path = DATA_DIR / f
    print(f"{f}: {'존재함' if file_path.exists() else '없음'}")

=== 파일 존재 여부 확인 ===
churn_logit1.csv: 없음
loan_default_logit2.csv: 없음


In [ ]:
if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"디렉토리 '{DATA_DIR}'를 생성했습니다.")
else:
    print(f"디렉토리 '{DATA_DIR}'가 이미 존재합니다.")

In [6]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path


In [11]:
DATA_DIR = Path('/content')
df = pd.read_csv(DATA_DIR / 'churn_logit1.csv')


In [12]:
print("=== 데이터 미리보기 ===")
print(df.head())

=== 데이터 미리보기 ===
   churn  age  usage_hour  complaint_cnt
0      0   29       12.07              0
1      0   20       70.05              2
2      0   23       58.09              0
3      0   37       67.25              2
4      0   54       78.19              0


In [13]:
print("\n=== 기본 정보 ===")
print(df.info())



=== 기본 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   churn          500 non-null    int64  
 1   age            500 non-null    int64  
 2   usage_hour     500 non-null    float64
 3   complaint_cnt  500 non-null    int64  
dtypes: float64(1), int64(3)
memory usage: 15.8 KB
None


In [14]:
model = smf.logit('churn ~ age + usage_hour + complaint_cnt', data=df).fit()


Optimization terminated successfully.
         Current function value: 0.237853
         Iterations 7


In [15]:
print("\n=== 로지스틱 회귀 요약 ===")
print(model.summary())


=== 로지스틱 회귀 요약 ===
                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                  500
Model:                          Logit   Df Residuals:                      496
Method:                           MLE   Df Model:                            3
Date:                Wed, 27 May 2026   Pseudo R-squ.:                  0.1754
Time:                        04:23:08   Log-Likelihood:                -118.93
converged:                       True   LL-Null:                       -144.22
Covariance Type:            nonrobust   LLR p-value:                 6.016e-11
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -1.2848      0.694     -1.852      0.064      -2.645       0.075
age              -0.0234      0.014     -1.615      0.106      -0.052       0.005
usage_hour       -0.

In [16]:
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)
})


In [17]:
print("\n=== 계수 / p-value / 오즈비 ===")
print(result_table)



=== 계수 / p-value / 오즈비 ===
                   coef       p_value  odds_ratio
Intercept     -1.284755  6.409071e-02    0.276718
age           -0.023406  1.063632e-01    0.976865
usage_hour    -0.038432  2.147205e-05    0.962297
complaint_cnt  0.656089  2.591138e-07    1.927240


In [18]:
conf = model.conf_int() #interval
conf.columns = ['2.5%', '97.5%']
conf['OR_2.5%'] = np.exp(conf['2.5%'])   #odd비는 exp
conf['OR_97.5%'] = np.exp(conf['97.5%'])



In [19]:
print("\n=== 계수 신뢰구간 및 오즈비 신뢰구간 ===")
print(conf)




=== 계수 신뢰구간 및 오즈비 신뢰구간 ===
                   2.5%     97.5%   OR_2.5%  OR_97.5%
Intercept     -2.644738  0.075228  0.071024  1.078130
age           -0.051817  0.005004  0.949503  1.005016
usage_hour    -0.056160 -0.020704  0.945388  0.979509
complaint_cnt  0.406446  0.905732  1.501471  2.473743


In [20]:
df['pred_prob'] = model.predict(df)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int) #결과를 정수로


In [21]:
print("\n=== 예측확률 상위 5개 ===")
print(df[['churn', 'pred_prob', 'pred_class']].head())



=== 예측확률 상위 5개 ===
   churn  pred_prob  pred_class
0      0   0.081106           0
1      0   0.041772           0
2      0   0.017030           0
3      0   0.031579           0
4      0   0.003858           0


In [22]:
print("\n=== 해석 가이드 ===")
for var in ['age', 'usage_hour', 'complaint_cnt']:
    coef = model.params[var]
    pval = model.pvalues[var]
    or_val = np.exp(coef)
    direction = '증가' if coef > 0 else '감소'
    print(f"{var}: 계수={coef:.4f}, p-value={pval:.6f}, 오즈비={or_val:.4f}")
    print(f" -> {var}가 1단위 증가할 때 이탈 odds는 약 {or_val:.4f}배가 되며, 방향은 {direction}입니다.")


=== 해석 가이드 ===
age: 계수=-0.0234, p-value=0.106363, 오즈비=0.9769
 -> age가 1단위 증가할 때 이탈 odds는 약 0.9769배가 되며, 방향은 감소입니다.
usage_hour: 계수=-0.0384, p-value=0.000021, 오즈비=0.9623
 -> usage_hour가 1단위 증가할 때 이탈 odds는 약 0.9623배가 되며, 방향은 감소입니다.
complaint_cnt: 계수=0.6561, p-value=0.000000, 오즈비=1.9272
 -> complaint_cnt가 1단위 증가할 때 이탈 odds는 약 1.9272배가 되며, 방향은 증가입니다.


In [23]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score
from pathlib import Path


In [24]:
DATA_DIR = Path('/content')
df = pd.read_csv(DATA_DIR / 'loan_default_logit2.csv')


In [25]:
print("=== 데이터 미리보기 ===")
print(df.head())
print("\n=== 기본 정보 ===")
print(df.info())



=== 데이터 미리보기 ===
   default   income  debt_ratio  late_cnt
0        0  3313.39       0.265         2
1        0  4026.60       0.305         0
2        0  3262.68       0.684         2
3        0  4087.12       1.031         0
4        0  2397.14       0.178         0

=== 기본 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   default     600 non-null    int64  
 1   income      600 non-null    float64
 2   debt_ratio  600 non-null    float64
 3   late_cnt    600 non-null    int64  
dtypes: float64(2), int64(2)
memory usage: 18.9 KB
None


In [26]:
model = smf.logit('default ~ income + debt_ratio + late_cnt', data=df).fit()



Optimization terminated successfully.
         Current function value: 0.426514
         Iterations 6


In [27]:
print("\n=== 로지스틱 회귀 요약 ===")
print(model.summary())




=== 로지스틱 회귀 요약 ===
                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                  600
Model:                          Logit   Df Residuals:                      596
Method:                           MLE   Df Model:                            3
Date:                Wed, 27 May 2026   Pseudo R-squ.:                  0.1477
Time:                        04:25:16   Log-Likelihood:                -255.91
converged:                       True   LL-Null:                       -300.24
Covariance Type:            nonrobust   LLR p-value:                 4.238e-19
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.7772      0.413     -4.302      0.000      -2.587      -0.967
income        -0.0003   5.94e-05     -5.422      0.000      -0.000      -0.000
debt_ratio     2.0093      0.370

In [28]:
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)   #오즈비는 exp
})


In [29]:
print("\n=== 계수 / p-value / 오즈비 ===")
print(result_table)



=== 계수 / p-value / 오즈비 ===
                coef       p_value  odds_ratio
Intercept  -1.777208  1.695477e-05    0.169110
income     -0.000322  5.892164e-08    0.999678
debt_ratio  2.009343  5.554017e-08    7.458417
late_cnt    0.477399  2.146767e-06    1.611877


In [30]:
llf = model.llf
residual_deviance = -2 * llf




In [31]:
print("\n=== 적합도 지표 ===")
print(f"log-likelihood (llf): {llf:.4f}")
print(f"residual deviance: {residual_deviance:.4f}")



=== 적합도 지표 ===
log-likelihood (llf): -255.9086
residual deviance: 511.8172


In [32]:
df['pred_prob'] = model.predict(df)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int)


In [33]:
acc = accuracy_score(df['default'], df['pred_class'])
err = 1 - acc


In [34]:
print("\n=== 분류 성능 ===")
print(f"accuracy     : {acc:.4f}")
print(f"error rate   : {err:.4f}")



=== 분류 성능 ===
accuracy     : 0.8150
error rate   : 0.1850


In [35]:
print("\n=== 예측확률 상위 10개 ===")
print(df[['default', 'pred_prob', 'pred_class']].head(10))


=== 예측확률 상위 10개 ===
   default  pred_prob  pred_class
0        0   0.204746           0
1        0   0.078639           0
2        0   0.377859           0
3        0   0.264700           0
4        0   0.100520           0
5        0   0.182376           0
6        0   0.242458           0
7        0   0.110946           0
8        0   0.158177           0
9        1   0.487071           0


In [36]:
conf = model.conf_int()
conf.columns = ['2.5%', '97.5%']
conf['OR_2.5%'] = np.exp(conf['2.5%'])
conf['OR_97.5%'] = np.exp(conf['97.5%'])


In [37]:
print("\n=== 계수 신뢰구간 및 오즈비 신뢰구간 ===")
print(conf)





=== 계수 신뢰구간 및 오즈비 신뢰구간 ===
                2.5%     97.5%   OR_2.5%   OR_97.5%
Intercept  -2.586962 -0.967453  0.075248   0.380050
income     -0.000438 -0.000206  0.999562   0.999794
debt_ratio  1.284415  2.734271  3.612555  15.398516
late_cnt    0.279960  0.674839  1.323076   1.963717


In [38]:
print("\n=== 해석 가이드 ===")
for var in ['income', 'debt_ratio', 'late_cnt']:
    coef = model.params[var]
    pval = model.pvalues[var]
    or_val = np.exp(coef)
    print(f"{var}: 계수={coef:.6f}, p-value={pval:.6f}, 오즈비={or_val:.6f}")





=== 해석 가이드 ===
income: 계수=-0.000322, p-value=0.000000, 오즈비=0.999678
debt_ratio: 계수=2.009343, p-value=0.000000, 오즈비=7.458417
late_cnt: 계수=0.477399, p-value=0.000002, 오즈비=1.611877
